# Augment existing Question-Answer JSON pairs with OpenAI GPT3.5-Turbo
This notebook contains testing and automated code to augment existing question-answer (QA) pairs JSON data with OpenAI's GPT3.5-turbo model.

In [1]:
import numpy as np
import pandas as pd
import os
import time
import openai
import json
import random

from tqdm import tqdm

## Set API key

In [21]:
api_key = '<PUT OPENAI API KEY HERE>'

## Define functions

In [3]:
def get_response(api_key, prompt, max_tokens=None, temperature=0.7, model="gpt-3.5-turbo"):
    '''
    Purpose: to input raw text to OpenAI API GPT model to generate a response in JSON format
    @params api_key: input API key from OpenAI (str)
    @params prompt: input prompt of what we want GPT model to do (str)
    @params max_tokens: input max tokens for model to generate completion; default set to None, since automatically calculated in function
    @params temperature: input randomness/creativity of model; default is 0.7 (which is also the default used by OpenAI)
    @params model: input OpenAI model to use; default is gpt-3.5-turbo
    returns: a response object from the API
    '''
    if max_tokens == None:
        max_tokens = round(4096 - (len(prompt)/4))  # calculate max_tokens for completion; divide prompt by 4 since 1 token ~= 4 chars
    else:  # override if max_tokens parameter is specified
        max_tokens = max_tokens
    
    client = openai.OpenAI(
        api_key=api_key
    )
    
    messages = [{"role": "user", "content": prompt}]
    
    response = client.chat.completions.create(
        messages=messages,
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=None # provide a list of strings to stop generation at certain points
    )

    return response

## Prompt testing
Only expand and use this section if you want to test the prompt, otherwise **ignore this section**.

In [93]:
original_question = "Which common food items are dangerous for dogs?"
original_answer = "Grapes, gum or candy containing xylitol, chocolate, and foods that contain garlic or onions are dangerous for dogs."

prompt = f"Rewrite the following question '{original_question}' in 6 different way to express the same query and rewrite 1 question by including a background story. And based on the original question, rewrite the answer '{original_answer}' in a professional and caring tone in response to the question, and sympathize with the context of the question. And then output everything in JSON where 'questions' contains a list of the 6 questions, 'bg_question' contains the 1 question with background story as a list, and 'answer' contains the 1 rewritten answer as a list."
response = get_response(api_key, prompt, max_tokens=None, temperature=0.9, model="gpt-3.5-turbo")
response = response.choices[0].message.content
response = json.loads(response)

response

{'questions': ['What food items pose a danger to dogs?',
  'Which foods should dogs avoid for safety reasons?',
  'Which common food items can harm dogs?',
  'What are some foods that are toxic to dogs?',
  'Which foods are not safe for dogs to consume?',
  'What food items can be harmful to dogs?'],
 'bg_question': ["After a recent scare with my own dog, I'm curious - which common food items are dangerous for dogs?"],
 'answer': ['It is important to keep grapes, gum or candy containing xylitol, chocolate, and foods that contain garlic or onions away from dogs, as these items can be dangerous and harmful to their health. Please be mindful and cautious when feeding your furry friends to prevent any potential risks.']}

In [100]:
test = response['questions'] + response['bg_question']
test

['What food items pose a danger to dogs?',
 'Which foods should dogs avoid for safety reasons?',
 'Which common food items can harm dogs?',
 'What are some foods that are toxic to dogs?',
 'Which foods are not safe for dogs to consume?',
 'What food items can be harmful to dogs?',
 "After a recent scare with my own dog, I'm curious - which common food items are dangerous for dogs?"]

In [98]:
response['answer']

['It is important to keep grapes, gum or candy containing xylitol, chocolate, and foods that contain garlic or onions away from dogs, as these items can be dangerous and harmful to their health. Please be mindful and cautious when feeding your furry friends to prevent any potential risks.']

## Automated code

In [4]:
# show current path of user
print(f'You are currently in the path: {os.getcwd()}')

You are currently in the path: /Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments


**Set folder path variables**

In [10]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

filename = 'dahlia_batch.json'          # input name of json file that contains all the original data
savefilename = 'dahlia_augment_data.json'  # input name of json file to save new data to 
errorfilename = 'error_log.json'        # input name of json file to save error message encountered

### Run code -- do not need to edit any code below this
**Notes:**
* No need to create the save file name or error file name beforehand. If the files do not exist, the code will automatically create them.
* If the save and error files already exist, the code will keep appending them with new data.
* At the end of this code, should get a maximum of 2 new files
  1. the JSON save file (i.e. ./all_augment_data.json)
  2. the JSON error file, if any errors do occur (i.e. ./error_log.json)

In [6]:
# load original json data
f = open(f'{alldata_dir}/{filename}')
data = json.load(f)
f.close()

# generate augmented data
print(f'Starting to process {filename} with a total of {len(data)} data points..')
for i in tqdm(range(len(data))):
    try:
        original_question = data[i]['question']
        original_answer = data[i]['answer']

        # make API call to openai
        prompt = f"Rewrite the following question '{original_question}' in 6 different way to express the same query and rewrite 1 question by including a background story. And based on the original question, rewrite the answer '{original_answer}' in a professional and caring tone in response to the question, and sympathize with the context of the question. And then output everything in JSON where 'questions' contains a list of the 6 questions, 'bg_question' contains the 1 question with background story as a list, and 'answer' contains the 1 rewritten answer as a list."

        response = get_response(api_key, prompt, max_tokens=None, temperature=0.9, model="gpt-3.5-turbo")
        response = response.choices[0].message.content
        response = json.loads(response)

        # organize data to save
        json_data = {}
        json_data['original_question'] = original_question  # stored as str
        json_data['original_answer'] = original_answer      # stored as str
        json_data['augment_questions'] = response['questions'] + response['bg_question']  # stored as list; question w/ background is always the last element
        json_data['new_answer'] = response['answer'][0]     # stored as str

        save_data = [json_data]  # convert dict to list of dict

        # save data to file
        if os.path.isfile(f'{alldata_dir}/{savefilename}') == False:  # if file to save new data doesn't exist already
            with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
                json.dump(save_data, f, ensure_ascii=False, indent=4)
            f.close()

        else:  # if file to save new data does exist already
            # read/load existing data
            f = open(f'{alldata_dir}/{savefilename}')
            existing_data = json.load(f)
            f.close()

            # add new data
            existing_data += save_data

            # save data to file
            with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, ensure_ascii=False, indent=4)
            f.close()
            
    except Exception as ex:  # if encounter error
        # organize error message to save
        error_data = {}
        error_data['data_index'] = i  # tells you which data point encountered error during augmentation
        error_data['question'] = data[i]['question']  # original data question
        error_data['answer'] = data[i]['answer']      # original data answer
        error_data['error_msg'] = str(ex)  # error message received
        
        save_error_data = [error_data]
        
        # save error data to error log file
        if os.path.isfile(f'{alldata_dir}/{errorfilename}') == False:  # if error file to save error messages doesn't exist already
            with open(f'{alldata_dir}/{errorfilename}', 'w', encoding='utf-8') as f:
                json.dump(save_error_data, f, ensure_ascii=False, indent=4)
            f.close()

        else:  # if error file to save error messages does exist already
            # read/load existing data
            f = open(f'{alldata_dir}/{errorfilename}')
            existing_data = json.load(f)
            f.close()

            # add new error data
            existing_data += save_error_data

            # save data to file
            with open(f'{alldata_dir}/{errorfilename}', 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, ensure_ascii=False, indent=4)
            f.close()

        
# check if all original data points were able to get augmented
if os.path.isfile(f'{alldata_dir}/{savefilename}') == True:
    f = open(f'{alldata_dir}/{savefilename}')
    data = json.load(f)
    f.close()
else:
    data = []  # if save file doesn't exist, set as empty list

    
if os.path.isfile(f'{alldata_dir}/{errorfilename}') == True:
    f = open(f'{alldata_dir}/{errorfilename}')
    error_data = json.load(f)
    f.close()
else:
    error_data = []  # if error file doesn't exist, set as empty list
    
print(f'[COMPLETE] Completed augmenting a total of {len(data)} data points with {len(error_data)} data points skipped due to error.')

Starting to process dahlia_batch.json with a total of 5000 data points..


100%|█████████████████████████████████████| 5000/5000 [4:39:16<00:00,  3.35s/it]

[COMPLETE] Completed augmenting a total of 4519 data points with 481 data points skipped due to error.


In [7]:
# check total number of data points after augmentation
f = open(f'{alldata_dir}/{savefilename}')
data = json.load(f)
f.close()

original_cnt = 0 # initiate original data counter
augment_cnt = 0  # initiate augmented data counter

for x in data:
    original_cnt += 1
    augment_cnt += len(x['augment_questions'])
    
print(f'There is a total of {original_cnt} original questions and {augment_cnt} augmented questions.')
print(f'This gives a total of {original_cnt + augment_cnt} data points.\n')

if augment_cnt % 7 == 0:
    print(f'[GOOD] All data points/questions got 7 new augmented data points generated.')
else:
    print(f'[WARNING] Not all data points/questions got 7 new augmented data points generated.')

There is a total of 4519 original questions and 31643 augmented questions.
This gives a total of 36162 data points.

[WARNING] Not all data points/questions got 7 new augmented data points generated.


### Manually check for data points that didn't pass the following requirements
* Did not get exactly 7 augmented questions
* Did not augment at all due to error encountering

#### Checking data points that did not get exactly 7 augmented questions

In [11]:
f = open(f'{alldata_dir}/{savefilename}')
data = json.load(f)
f.close()

for i in range(len(data)):
    dat = data[i]
    aug_num = len(dat['augment_questions'])
    
    if aug_num == 7:
        pass
    else:
        print(f'Data point {i} had {aug_num} augmented questions; did not have 7')

Data point 154 had 8 augmented questions; did not have 7
Data point 272 had 8 augmented questions; did not have 7
Data point 303 had 8 augmented questions; did not have 7
Data point 405 had 8 augmented questions; did not have 7
Data point 730 had 8 augmented questions; did not have 7
Data point 785 had 8 augmented questions; did not have 7
Data point 2036 had 8 augmented questions; did not have 7
Data point 2231 had 8 augmented questions; did not have 7
Data point 2667 had 8 augmented questions; did not have 7
Data point 4240 had 8 augmented questions; did not have 7


In [21]:
# check total number of data points after fixing the data points
f = open(f'{alldata_dir}/{savefilename}')
data = json.load(f)
f.close()

original_cnt = 0 # initiate original data counter
augment_cnt = 0  # initiate augmented data counter

for x in data:
    original_cnt += 1
    augment_cnt += len(x['augment_questions'])
    
print(f'There is a total of {original_cnt} original questions and {augment_cnt} augmented questions.')
print(f'This gives a total of {original_cnt + augment_cnt} data points.\n')

if augment_cnt % 7 == 0:
    print(f'[GOOD] All data points/questions got 7 new augmented data points generated.')
else:
    print(f'[WARNING] Not all data points/questions got 7 new augmented data points generated.')

There is a total of 4519 original questions and 31633 augmented questions.
This gives a total of 36152 data points.

[GOOD] All data points/questions got 7 new augmented data points generated.


#### Reprocess data points that encountered error

In [32]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

filename = 'juyeon_error_log.json'          # input name of json file that contains all the original data
savefilename = 'juyeon_reprocess_data.json'  # input name of json file to save new data to 
errorfilename = 'juyeon_error_reprocess_log.json'        # input name of json file to save error message encountered

In [36]:
# load original json data
f = open(f'{alldata_dir}/{filename}')
data = json.load(f)
f.close()

# generate augmented data
print(f'Starting to process {filename} with a total of {len(data)} data points..')
for i in tqdm(range(len(data))):
    try:
        original_question = data[i]['question']
        original_answer = data[i]['answer']

        # make API call to openai
        prompt = f"Rewrite the following question '{original_question}' in 6 different way to express the same query and rewrite 1 question by including a background story. And based on the original question, rewrite the answer '{original_answer}' in a professional and caring tone in response to the question, and sympathize with the context of the question. And then output everything in JSON where 'questions' contains a list of the 6 questions, 'bg_question' contains the 1 question with background story as a list, and 'answer' contains the 1 rewritten answer as a list."

        response = get_response(api_key, prompt, max_tokens=None, temperature=0.9, model="gpt-3.5-turbo")
        response = response.choices[0].message.content
        response = json.loads(response)

        # organize data to save
        json_data = {}
        json_data['original_question'] = original_question  # stored as str
        json_data['original_answer'] = original_answer      # stored as str
        json_data['augment_questions'] = response['questions'] + response['bg_question']  # stored as list; question w/ background is always the last element
        json_data['new_answer'] = response['answer'][0]     # stored as str

        save_data = [json_data]  # convert dict to list of dict

        # save data to file
        if os.path.isfile(f'{alldata_dir}/{savefilename}') == False:  # if file to save new data doesn't exist already
            with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
                json.dump(save_data, f, ensure_ascii=False, indent=4)
            f.close()

        else:  # if file to save new data does exist already
            # read/load existing data
            f = open(f'{alldata_dir}/{savefilename}')
            existing_data = json.load(f)
            f.close()

            # add new data
            existing_data += save_data

            # save data to file
            with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, ensure_ascii=False, indent=4)
            f.close()
            
    except Exception as ex:  # if encounter error
        # organize error message to save
        error_data = {}
        error_data['data_index'] = data[i]['data_index']  # data point index stored in error_log.json
        error_data['question'] = data[i]['question']  # original data question
        error_data['answer'] = data[i]['answer']      # original data answer
        error_data['error_msg'] = str(ex)  # error message received
        
        save_error_data = [error_data]
        
        # save error data to error log file
        if os.path.isfile(f'{alldata_dir}/{errorfilename}') == False:  # if error file to save error messages doesn't exist already
            with open(f'{alldata_dir}/{errorfilename}', 'w', encoding='utf-8') as f:
                json.dump(save_error_data, f, ensure_ascii=False, indent=4)
            f.close()

        else:  # if error file to save error messages does exist already
            # read/load existing data
            f = open(f'{alldata_dir}/{errorfilename}')
            existing_data = json.load(f)
            f.close()

            # add new error data
            existing_data += save_error_data

            # save data to file
            with open(f'{alldata_dir}/{errorfilename}', 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, ensure_ascii=False, indent=4)
            f.close()

        
# check if all original data points were able to get augmented
if os.path.isfile(f'{alldata_dir}/{savefilename}') == True:
    f = open(f'{alldata_dir}/{savefilename}')
    data = json.load(f)
    f.close()
else:
    data = []  # if save file doesn't exist, set as empty list

    
if os.path.isfile(f'{alldata_dir}/{errorfilename}') == True:
    f = open(f'{alldata_dir}/{errorfilename}')
    error_data = json.load(f)
    f.close()
else:
    error_data = []  # if error file doesn't exist, set as empty list
    
print(f'[COMPLETE] Completed augmenting a total of {len(data)} data points with {len(error_data)} data points skipped due to error.')

Starting to process juyeon_error_log.json with a total of 1 data points..


100%|████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.89s/it]

[COMPLETE] Completed augmenting a total of 193 data points with 0 data points skipped due to error.


**Check total count between initial augmentation and error reprocess**

In [16]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

filename = 'dahlia_augment_data.json'
reprocessfilename = 'dahlia_reprocess_data.json'

# load json data
f1 = open(f'{alldata_dir}/{filename}')
data1 = json.load(f1)
f1.close()

f2 = open(f'{alldata_dir}/{reprocessfilename}')
data2 = json.load(f2)
f1.close()

len(data1) + len(data2)

5001

In [25]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

file1 = 'gabriel_augment_data_1.json'
file2 = 'gabriel_augment_data_2.json'
reprocessfilename = 'gabriel_reprocess_data_2.json'

# load json data
f1 = open(f'{alldata_dir}/{file1}')
data1 = json.load(f1)
f1.close()

f2 = open(f'{alldata_dir}/{file2}')
data2 = json.load(f2)
f1.close()

f3 = open(f'{alldata_dir}/{reprocessfilename}')
data3 = json.load(f3)
f1.close()

len(data1) + len(data2) + len(data3)

5008

In [26]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

filename = 'deepak_augment_data.json'

# load json data
f1 = open(f'{alldata_dir}/{filename}')
data1 = json.load(f1)
f1.close()

len(data1)

5000

In [37]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

filename = 'sungjun_augment_data.json'
reprocessfilename = 'sungjun_reprocess_data.json'

# load json data
f1 = open(f'{alldata_dir}/{filename}')
data1 = json.load(f1)
f1.close()

f2 = open(f'{alldata_dir}/{reprocessfilename}')
data2 = json.load(f2)
f1.close()

len(data1) + len(data2)

5000

In [38]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

filename = 'juyeon_augment_data.json'
reprocessfilename = 'juyeon_reprocess_data.json'

# load json data
f1 = open(f'{alldata_dir}/{filename}')
data1 = json.load(f1)
f1.close()

f2 = open(f'{alldata_dir}/{reprocessfilename}')
data2 = json.load(f2)
f1.close()

len(data1) + len(data2)

4538

**Manually check and reprocess data points**

In [48]:
i = 3

original_question = errors[i]['question']
original_answer = errors[i]['answer']

# make API call to openai
prompt = f"Rewrite the following question '{original_question}' in 6 different way to express the same query and rewrite 1 question by including a background story. And based on the original question, rewrite the answer '{original_answer}' in a professional and caring tone in response to the question, and sympathize with the context of the question. And then output everything in JSON where 'questions' contains a list of the 6 questions, 'bg_question' contains the 1 question with background story as a list, and 'answer' contains the 1 rewritten answer as a list."

response = get_response(api_key, prompt, max_tokens=None, temperature=0.9, model="gpt-3.5-turbo")
response = response.choices[0].message.content
response = json.loads(response)
response

{'questions': ['What is the importance of socialization for puppies?',
  'How does socialization benefit puppies?',
  'Why do puppies need socialization?',
  'In what ways does socialization impact puppies?',
  "What role does socialization play in a puppy's development?",
  'What are the advantages of socializing puppies?'],
 'bg_question': ['When I first brought home my new puppy, he seemed scared of everything outside of our house. Why is socialization important for puppies?'],
 'answer': ['Socialization is crucial for puppies as it helps them learn that encountering new and different situations does not necessarily have to be a negative or fearful experience. It is essential for their overall development and well-being.']}

In [49]:
# organize data to save
json_data = {}
json_data['original_question'] = original_question  # stored as str
json_data['original_answer'] = original_answer      # stored as str
json_data['augment_questions'] = response['questions'] + response['bg_question']  # stored as list; question w/ background is always the last element
json_data['new_answer'] = response['answer'][0]     # stored as str

save_data = [json_data]  # convert dict to list of dict
save_data

[{'original_question': 'Why is socialization important for puppies?',
  'original_answer': "Socialization is important for puppies because it helps them understand that 'new and different' doesn’t necessarily mean 'bad and scary'.",
  'augment_questions': ['What is the importance of socialization for puppies?',
   'How does socialization benefit puppies?',
   'Why do puppies need socialization?',
   'In what ways does socialization impact puppies?',
   "What role does socialization play in a puppy's development?",
   'What are the advantages of socializing puppies?',
   'When I first brought home my new puppy, he seemed scared of everything outside of our house. Why is socialization important for puppies?'],
  'new_answer': 'Socialization is crucial for puppies as it helps them learn that encountering new and different situations does not necessarily have to be a negative or fearful experience. It is essential for their overall development and well-being.'}]

In [50]:
# check if there are 7 augmented questions
if len(save_data[0]['augment_questions']) == 7:
    print(f"[PASS] Has 7 augmented questions")
else:
    print(f"[FAIL] Has {len(save_data[0]['augment_questions'])} augmented questions.")

[PASS] Has 7 augmented questions


In [51]:
# save data to file
if os.path.isfile(f'{alldata_dir}/{savefilename}') == False:  # if file to save new data doesn't exist already
    with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
        json.dump(save_data, f, ensure_ascii=False, indent=4)
    f.close()

else:  # if file to save new data does exist already
    # read/load existing data
    f = open(f'{alldata_dir}/{savefilename}')
    existing_data = json.load(f)
    f.close()

    # add new data
    existing_data += save_data

    # save data to file
    with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
        json.dump(existing_data, f, ensure_ascii=False, indent=4)
    f.close()

In [52]:
# check total number of data points after augmentation
f = open(f'{alldata_dir}/{savefilename}')
data = json.load(f)
f.close()

original_cnt = 0 # initiate original data counter
augment_cnt = 0  # initiate augmented data counter

for x in data:
    original_cnt += 1
    augment_cnt += len(x['augment_questions'])
    
print(f'There is a total of {original_cnt} original questions and {augment_cnt} augmented questions.')
print(f'This gives a total of {original_cnt + augment_cnt} data points.\n')

if augment_cnt % 7 == 0:
    print(f'[GOOD] All data points/questions got 7 new augmented data points generated.')
else:
    print(f'[WARNING] Not all data points/questions got 7 new augmented data points generated.')

There is a total of 4523 original questions and 31661 augmented questions.
This gives a total of 36184 data points.

[GOOD] All data points/questions got 7 new augmented data points generated.


There is a total of 4519 original questions and 31643 augmented questions.

## Augment Gabriel data subset [2500:5000]

In [4]:
# show current path of user
print(f'You are currently in the path: {os.getcwd()}')

You are currently in the path: /Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments


In [5]:
# input path of directory of where 'all_data.json' file is located -- this directory will store new data and error file as well
alldata_dir = '/Users/dahliama/Desktop/data298/github/projectmarley/data/data-preprocess/03_data_augments'

filename = 'gabriel_batch.json'          # input name of json file that contains all the original data
savefilename = 'gabriel_augment_data_2.json'  # input name of json file to save new data to 
errorfilename = 'gabriel_error_log_2.json'        # input name of json file to save error message encountered

In [6]:
# load original json data
f = open(f'{alldata_dir}/{filename}')
data = json.load(f)
f.close()

# augment for last half of Gabriel's data subset
data = data[2500:5000]

# generate augmented data
print(f'Starting to process {filename} with a total of {len(data)} data points..')
for i in tqdm(range(len(data))):
    try:
        original_question = data[i]['question']
        original_answer = data[i]['answer']

        # make API call to openai
        prompt = f"Rewrite the following question '{original_question}' in 6 different way to express the same query and rewrite 1 question by including a background story. And based on the original question, rewrite the answer '{original_answer}' in a professional and caring tone in response to the question, and sympathize with the context of the question. And then output everything in JSON where 'questions' contains a list of the 6 questions, 'bg_question' contains the 1 question with background story as a list, and 'answer' contains the 1 rewritten answer as a list."

        response = get_response(api_key, prompt, max_tokens=None, temperature=0.9, model="gpt-3.5-turbo")
        response = response.choices[0].message.content
        response = json.loads(response)

        # organize data to save
        json_data = {}
        json_data['original_question'] = original_question  # stored as str
        json_data['original_answer'] = original_answer      # stored as str
        json_data['augment_questions'] = response['questions'] + response['bg_question']  # stored as list; question w/ background is always the last element
        json_data['new_answer'] = response['answer'][0]     # stored as str

        save_data = [json_data]  # convert dict to list of dict

        # save data to file
        if os.path.isfile(f'{alldata_dir}/{savefilename}') == False:  # if file to save new data doesn't exist already
            with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
                json.dump(save_data, f, ensure_ascii=False, indent=4)
            f.close()

        else:  # if file to save new data does exist already
            # read/load existing data
            f = open(f'{alldata_dir}/{savefilename}')
            existing_data = json.load(f)
            f.close()

            # add new data
            existing_data += save_data

            # save data to file
            with open(f'{alldata_dir}/{savefilename}', 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, ensure_ascii=False, indent=4)
            f.close()
            
    except Exception as ex:  # if encounter error
        # organize error message to save
        error_data = {}
        error_data['data_index'] = i  # tells you which data point encountered error during augmentation
        error_data['question'] = data[i]['question']  # original data question
        error_data['answer'] = data[i]['answer']      # original data answer
        error_data['error_msg'] = str(ex)  # error message received
        
        save_error_data = [error_data]
        
        # save error data to error log file
        if os.path.isfile(f'{alldata_dir}/{errorfilename}') == False:  # if error file to save error messages doesn't exist already
            with open(f'{alldata_dir}/{errorfilename}', 'w', encoding='utf-8') as f:
                json.dump(save_error_data, f, ensure_ascii=False, indent=4)
            f.close()

        else:  # if error file to save error messages does exist already
            # read/load existing data
            f = open(f'{alldata_dir}/{errorfilename}')
            existing_data = json.load(f)
            f.close()

            # add new error data
            existing_data += save_error_data

            # save data to file
            with open(f'{alldata_dir}/{errorfilename}', 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, ensure_ascii=False, indent=4)
            f.close()

        
# check if all original data points were able to get augmented
if os.path.isfile(f'{alldata_dir}/{savefilename}') == True:
    f = open(f'{alldata_dir}/{savefilename}')
    data = json.load(f)
    f.close()
else:
    data = []  # if save file doesn't exist, set as empty list

    
if os.path.isfile(f'{alldata_dir}/{errorfilename}') == True:
    f = open(f'{alldata_dir}/{errorfilename}')
    error_data = json.load(f)
    f.close()
else:
    error_data = []  # if error file doesn't exist, set as empty list
    
print(f'[COMPLETE] Completed augmenting a total of {len(data)} data points with {len(error_data)} data points skipped due to error.')

Starting to process gabriel_batch.json with a total of 2500 data points..


100%|█████████████████████████████████████| 2500/2500 [2:16:32<00:00,  3.28s/it]


[COMPLETE] Completed augmenting a total of 2382 data points with 118 data points skipped due to error.


In [7]:
# check total number of data points after augmentation
f = open(f'{alldata_dir}/{savefilename}')
data = json.load(f)
f.close()

original_cnt = 0 # initiate original data counter
augment_cnt = 0  # initiate augmented data counter

for x in data:
    original_cnt += 1
    augment_cnt += len(x['augment_questions'])
    
print(f'There is a total of {original_cnt} original questions and {augment_cnt} augmented questions.')
print(f'This gives a total of {original_cnt + augment_cnt} data points.\n')

if augment_cnt % 7 == 0:
    print(f'[GOOD] All data points/questions got 7 new augmented data points generated.')
else:
    print(f'[WARNING] Not all data points/questions got 7 new augmented data points generated.')

There is a total of 2382 original questions and 16687 augmented questions.
This gives a total of 19069 data points.

[WARNING] Not all data points/questions got 7 new augmented data points generated.
